In [2]:
from scripts.tvb_nest_script import *
from tvb_multiscale.core.plot.plotter import Plotter
from tvb.contrib.scripts.datatypes.time_series_xarray import TimeSeriesRegion as TimeSeriesXarray

from scripts.nest_script import *        #build_NEST_network, plot_nest_results

model_params = {'STIMULUS': 0.1, 'G': 6}        # Tuning is done at baseline

tuned_values_tvb_nest = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]

tuned_values_nest_tvb = [100, 150]

tuned_value_nest_tvb = 122   # Computed from NEST only sim of 37.5 sec with 7.5 transient (as cosims)
tuned_value_tvb_nest = 0.04375  #0.65          # 0.8 
seed = 10
test_name = 'cosim'         # 'cosim', 'tvb-only', 'cerebOFF'
if test_name=='cosim':
    COMPUTE_REF = False          # True if you want to run TVB-only
    CEREB_OFF = False
elif test_name=='tvb-only':
    COMPUTE_REF = True
    CEREB_OFF = False
elif test_name == 'cerebOFF':
    COMPUTE_REF = True
    CEREB_OFF = True

path = 'tuned_interfaces_max_rate_'+str(tuned_value_nest_tvb)+'_rnd'+str(seed)+'_'+test_name
# Get configuration
config, plotter = configure(output_folder=path, verbose=2)
print("config.NEST_PERIPHERY",config.NEST_PERIPHERY)
config.model_params.update(model_params)
config.SIMULATION_LENGTH = 30000 #20000 30000
config.TRANSIENT_RATIO = 0.25
config.RANDOM_SEED_TVB = seed
config.RANDOM_SEED_NEST = seed
# Load and prepare connectome and connectivity with all possible normalizations:
connectome, major_structs_labels, voxel_count, inds, maps = prepare_connectome(config, plotter=plotter)
connectivity = build_connectivity(connectome, inds, config)

# Scale up connections from principal sensory trigeminal nucleus to ansiform lobule
reg1='Left Ansiform lobule'
reg2='Right Ansiform lobule'
reg3 = 'Left Principal sensory nucleus of the trigeminal'
reg4 = 'Right Principal sensory nucleus of the trigeminal'
reg5 = 'Left Spinal nucleus of the trigeminal'
reg6 = 'Right Spinal nucleus of the trigeminal'
#find the indices in region labels of these strings
iR1 = np.where([reg1 in reg for reg in connectivity.region_labels])[0]
iR2 = np.where([reg2 in reg for reg in connectivity.region_labels])[0]
iR3 = np.where([reg3 in reg for reg in connectivity.region_labels])[0]
iR4 = np.where([reg4 in reg for reg in connectivity.region_labels])[0]
iR5 = np.where([reg5 in reg for reg in connectivity.region_labels])[0]
iR6 = np.where([reg6 in reg for reg in connectivity.region_labels])[0]

pathway_gain = 1
'''# PST to AN
connectivity.weights[iR1, iR3] *= pathway_gain      
connectivity.weights[iR1, iR4] *= pathway_gain    
connectivity.weights[iR2, iR3] *= pathway_gain      
connectivity.weights[iR2, iR4] *= pathway_gain
# SNT to PST
connectivity.weights[iR3, iR5] *= pathway_gain      
connectivity.weights[iR3, iR6] *= pathway_gain    
connectivity.weights[iR4, iR5] *= pathway_gain      
connectivity.weights[iR4, iR6] *= pathway_gain
'''
# SNT to AN
connectivity.weights[iR1, iR5] *= pathway_gain      
connectivity.weights[iR1, iR6] *= pathway_gain    
connectivity.weights[iR2, iR5] *= pathway_gain      
connectivity.weights[iR2, iR6] *= pathway_gain

# To have the full sensory whisking pathway
reg7 = 'Left Primary somatosensory area, barrel field'
reg8 = 'Right Primary somatosensory area, barrel field'
iR7 = np.where([reg7 in reg for reg in connectivity.region_labels])[0]
iR8 = np.where([reg8 in reg for reg in connectivity.region_labels])[0]
# S1 to PST
connectivity.weights[iR3, iR7] *= pathway_gain      
connectivity.weights[iR3, iR8] *= pathway_gain    
connectivity.weights[iR4, iR7] *= pathway_gain      
connectivity.weights[iR4, iR8] *= pathway_gain
# SNT to S1
connectivity.weights[iR7, iR5] *= pathway_gain      
connectivity.weights[iR7, iR6] *= pathway_gain    
connectivity.weights[iR8, iR5] *= pathway_gain      
connectivity.weights[iR8, iR6] *= pathway_gain


# Put cereb weights to 0 if CEREB_OFF
if CEREB_OFF:
    #reg1='Cerebell*'
    reg1='Left Cerebellar Cortex'
    reg2='Left Cerebellar Nuclei'
    reg3='Left Ansiform lobule'
    reg4='Left Interposed nucleus'
    reg5='Right Cerebellar Cortex'
    reg6='Right Cerebellar Nuclei'
    reg7='Right Ansiform lobule'
    reg8='Right Interposed nucleus'
    #find the indices in region labels of these strings
    iR1 = np.where([reg1 in reg for reg in connectivity.region_labels])[0]
    iR2 = np.where([reg2 in reg for reg in connectivity.region_labels])[0]
    iR3 = np.where([reg3 in reg for reg in connectivity.region_labels])[0]
    iR4 = np.where([reg4 in reg for reg in connectivity.region_labels])[0]
    iR5 = np.where([reg5 in reg for reg in connectivity.region_labels])[0]
    iR6 = np.where([reg6 in reg for reg in connectivity.region_labels])[0]
    iR7 = np.where([reg7 in reg for reg in connectivity.region_labels])[0]
    iR8 = np.where([reg8 in reg for reg in connectivity.region_labels])[0]
    # for reg1, reg2, sc in config.BRAIN_CONNECTIONS_TO_SCALE:
    #     iR1 = np.where([reg in reg1 for reg in connectivity.region_labels])[0]
    #     iR2 = np.where([reg in reg2 for reg in connectivity.region_labels])[0]
    #     connectivity.weights[iR1, iR2] *= 0
    #iR1
    connectivity.weights.shape
    for i in [iR1, iR2, iR3, iR4, iR5, iR6, iR7, iR8]:
        connectivity.weights[i,:]=0
        connectivity.weights[:,i]=0
                        # , iR2, iR3, iR4, iR5, iR6, iR7, iR8] = 0
    connectivity.weights[iR1,:]



# Prepare model
model = build_model(connectivity.number_of_regions, inds, maps, config)
# Prepare simulator
simulator = build_simulator(connectivity, model, inds, maps, config, plotter=plotter)

if COMPUTE_REF:
    # Run simulation and get results for reference values
    results, transient = simulate(simulator, config)
else:
    # Build TVB-NEST interfaces
    nest_network, nest_nodes_inds, neuron_models, neuron_number = build_NEST_network(config)
    simulator, nest_network = build_tvb_nest_interfaces(simulator, nest_network, nest_nodes_inds, config, tvb_to_nest_gain=tuned_value_tvb_nest, max_rate_to_tune=tuned_value_nest_tvb)
    # Simulate TVB-NEST model
    results, transient, simulator, nest_network = simulate_tvb_nest(simulator, nest_network, config)
        
print(results)
       

        
# Target values: ansilob=-0.3263, interposed=-0.3209, oliv=-0.3284




ModuleNotFoundError: No module named 'examples.tvb_nest.notebooks.cerebellum'

<Figure size 432x288 with 0 Axes>

In [ ]:
# Save results
path = 'tuned_interfaces_max_rate_'+str(tuned_value_nest_tvb)+'_rnd'+str(seed)+'_'+test_name
print(results)
import pickle
with open('results_cerebON_cosim_30sec_CORRECT'+str(seed)+'_'+test_name+'.pickle', 'wb') as handle:
   pickle.dump(results, handle)

In [ ]:
# Compute coherence

path = '/outputs/tuned_interfaces_max_rate_'+str(tuned_value_nest_tvb)+'_rnd'+str(seed)+'_'+test_name
transient = config.TRANSIENT_RATIO * config.SIMULATION_LENGTH
if config.RAW_PERIOD > config.DEFAULT_DT:
    transient = (transient // config.RAW_PERIOD) * config.RAW_PERIOD + config.RAW_PERIOD/2
import pickle
print(os.getcwd() + path+'/results_cerebON_cosim_30sec_CORRECT'+str(seed)+'_'+test_name+'.pickle')
results = pickle.load(open(os.getcwd() + path+'/results_cerebON_cosim_30sec_CORRECT'+str(seed)+'_'+test_name+'.pickle', 'rb'))
source_ts, CxyR, fR, fL, CxyL = plot_tvb(transient, inds, results=results, source_ts=None, bold_ts=None, afferent_ts=None,
                    simulator=simulator, plotter=plotter, config=config, write_files=True)


In [ ]:
# Save coherence
with open(os.getcwd() + path+'coherence_30sec_rnd'+str(seed)+'_'+test_name+'.pickle', 'wb') as handle:
    pickle.dump([CxyR, fR, fL, CxyL], handle)

In [ ]:
# Load coherence files and plot

[CxyR_cerebOFF1, fR_cerebOFF1, fL_cerebOFF1, CxyL_cerebOFF1] = pickle.load(open('coherence_MF_cerebOFF_30sec.pickle', 'rb'))
[CxyR_cerebON_MF1, fR_cerebON_MF1, fL_cerebON_MF1, CxyL_cerebON_MF1] = pickle.load(open('coherence_MF_cerebON_MF_30sec.pickle', 'rb'))
[CxyR_cerebON1, fR_cerebON1, fL_cerebON1, CxyL_cerebON1] = pickle.load(open('coherence_MF_cerebON_30sec.pickle', 'rb'))

[CxyR_cerebOFF2, fR_cerebOFF2, fL_cerebOFF2, CxyL_cerebOFF2] = pickle.load(open('coherence_MF_cerebOFF_30sec.pickle', 'rb'))
[CxyR_cerebON_MF2, fR_cerebON_MF2, fL_cerebON_MF2, CxyL_cerebON_MF2] = pickle.load(open('coherence_MF_cerebON_MF_30sec.pickle', 'rb'))
[CxyR_cerebON2, fR_cerebON2, fL_cerebON2, CxyL_cerebON2] = pickle.load(open('coherence_MF_cerebON_30sec.pickle', 'rb'))

# Concatenate
Cxy_cerebOFF_all = np.vstack((CxyR_cerebOFF1,CxyL_cerebOFF1,CxyR_cerebOFF2,CxyL_cerebOFF2))
Cxy_cerebON_MF_all = np.vstack((CxyR_cerebON_MF1,CxyL_cerebON_MF1,CxyR_cerebON_MF2,CxyL_cerebON_MF2))
Cxy_cerebON_all = np.vstack((CxyR_cerebON1,CxyL_cerebON1,CxyR_cerebON2,CxyL_cerebON2))

# Plot average
CxyR_cerebOFF_avg = np.mean(Cxy_cerebOFF_all)
CxyR_cerebON_MF_avg = np.mean(Cxy_cerebON_MF_all)
CxyR_cerebON_avg = np.mean(Cxy_cerebON_all)


plt.plot(fR_cerebOFF1,CxyR_cerebOFF_avg)
plt.plot(fR_cerebOFF1,CxyR_cerebON_MF_avg)
plt.plot(fR_cerebOFF1,CxyR_cerebON_avg)

In [ ]:
# Compute baseline of TVB regions of interest
regs = ['ansilob', 'interposed', 'oliv']            # L and R ? 

# results=None

# Compute error
import pickle, math
ref_psd_from_file = pickle.load(open('ref_psd_mossy.pkl','rb'))
ref_mossy_firing = pickle.load(open('ref_firing_rate_mossy.pkl','rb'))

# NEW CODE after Denis' refactoring
afferent_ts = results[-1]
ansilob_afferent_coupling_psd_rmse(ref_mossy_firing, afferent_ts, inds, path_gain=pathway_gain, ftarg=None, transient=transient)


# Plot
source_ts, Pxx_den_ansilob = plot_tvb(transient, inds, results=results, source_ts=None, bold_ts=None, afferent_ts=None,
                    simulator=simulator, plotter=plotter, config=config, write_files=True)



# OLD CODE:
# Compute ref PSD from mossy firing rate
#fs = 1
#from scipy import signal
#f_ref, Pxx_den_ref = signal.welch(ref_mossy_firing, fs, nperseg=2048)
# freq from 5 to 47 (config.TARGET_FREQS = np.arange(5.0, 48.0, 1.0))

# NB: use compute_target_PSD (1D and from raw data)! to have the TVB value!! Pxx_den_ansilob = compute_target_etc!!! (Check on rising-net, pull last!!)


# Adding the time vector to ref_mossy_firing - for a sim duration of 10s and 5-ms time bins
Pxx_den_ref, f_ref = compute_data_PSDs_from_nest([list(np.linspace(2.5, 9997.5, 2000)),ref_mossy_firing])
MSE = np.square(np.subtract(Pxx_den_ansilob,Pxx_den_ref)).mean() 
RMSE = math.sqrt(MSE)
print("RMSEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE with pathway gain = ", pathway_gain, " is ",RMSE)

# Use compute_data_PSDs_1D
PSD_Popa = compute_target_PSDs_1D(config)
PSD_target_ansilob = {"f": PSD_Popa['f'], "PSD_target": Pxx_den_ref}
PSD_actual_ansilob = compute_data_PSDs_1D(results[0], PSD_target=PSD_target_ansilob)
if results is not None:
    source_ts = TimeSeriesXarray(  # substitute with TimeSeriesRegion fot TVB like functionality
            data=results[0][1], time=results[0][0],
            connectivity=simulator.connectivity,
            labels_ordering=["Time", "State Variable", "Region", "Neurons"],
            labels_dimensions={"State Variable": list(simulator.model.variables_of_interest),
                               "Region": simulator.connectivity.region_labels.tolist()},
            sample_period=simulator.integrator.dt)
    source_ts.configure()

    t = source_ts.time


source_ts_interface = {}
#print(inds.keys(), inds['oliv'])
if source_ts is not None:
    for reg in regs:
        source_ts_interface[reg] = source_ts[-config.SIMULATION_LENGTH:, 0, inds[reg][1]]
        #print("source ts shape ", source_ts_interface[reg])
        #print("source ts parts", source_ts_interface[reg].Time, source_ts_interface[reg].values)
        print("Avg baseline for ", reg, np.mean(source_ts_interface[reg]))
        

In [ ]:
e,d=plot_tvb(transient, inds, results=results, source_ts=None, bold_ts=None, afferent_ts=None,
                       simulator=simulator, plotter=plotter, config=config, write_files=True)

In [ ]:
print(config.figures.SAVE_FLAG)
config.figures.SAVE_FLAG=False
from examples.tvb_nest.notebooks.cerebellum.scripts.nest_script import plot_nest_results_raster
plot_nest_results_raster(nest_network, neuron_models, neuron_number, config)